# Module 20 Lab — CrewAI: Content Production Pipeline with Hierarchical Crew and Flows

**Goal:** Build a two-stage content production pipeline using CrewAI's two process types and a Flow with a quality gate router.

Architecture overview:
```
Flow.research()  →  research_crew (sequential)  →  5 verified facts
Flow.write()     →  writing_crew (hierarchical)  →  draft article
Flow.quality_gate()  →  router
    ├── "publish"  →  Flow.publish()   (draft > 150 words)
    └── "revise"   →  Flow.revise()    (draft too short)
```

By the end of this lab you will be able to:
- Configure `Process.sequential` vs `Process.hierarchical`
- Wire tasks with `context=` dependencies
- Implement a `Flow` with `@start`, `@listen`, and `@router` decorators
- Read `usage_metrics` to compare token costs across crews

In [ ]:
!pip install crewai crewai-tools langchain-anthropic langchain-openai

In [ ]:
import os
from crewai import Agent, Task, Crew, Process
from crewai.flow.flow import Flow, start, listen, router

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

if ANTHROPIC_API_KEY:
    from langchain_anthropic import ChatAnthropic
    llm = ChatAnthropic(model="claude-haiku-4-5-20251001", api_key=ANTHROPIC_API_KEY)
    print("Using Anthropic: claude-haiku-4-5-20251001")
elif OPENAI_API_KEY:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY)
    print("Using OpenAI: gpt-4o-mini")
else:
    raise ValueError("Set ANTHROPIC_API_KEY or OPENAI_API_KEY in environment")

## Section 1 — Research Crew Agents

CrewAI agents have three identity fields that shape LLM behaviour:

| Field | Effect |
|---|---|
| `role` | Appears in the system prompt as the agent's job title |
| `goal` | The primary objective — used in task allocation |
| `backstory` | Persona detail that anchors tone and expertise level |

The research crew uses `Process.sequential`, so agents execute in list order: researcher first, then fact_checker.

**TODO 1:** Create the `researcher` agent with the parameters shown in the comment.

**TODO 2:** Create the `fact_checker` agent. Use `allow_delegation=False` and `max_iter=3`.

In [ ]:
# TODO 1: Create the researcher agent
# researcher = Agent(
#     role="Senior Researcher",
#     goal="Find accurate and comprehensive information about the given topic",
#     backstory="PhD researcher with 10 years experience in tech journalism. Known for finding primary sources.",
#     llm=llm,
#     allow_delegation=False,
#     verbose=True,
#     max_iter=3,
# )
researcher = None  # replace with Agent(...)

# TODO 2: Create the fact_checker agent
# fact_checker = Agent(
#     role="Fact Checker",
#     goal="Verify every claim against primary sources before it reaches the writer",
#     backstory="Meticulous editor with a background in investigative journalism. Marks each claim CONFIRMED or UNCERTAIN.",
#     llm=llm,
#     allow_delegation=False,
#     verbose=True,
#     max_iter=3,
# )
fact_checker = None  # replace with Agent(...)

# === SOLUTION (reveal only if stuck) ===
# researcher = Agent(
#     role="Senior Researcher",
#     goal="Find accurate and comprehensive information about the given topic",
#     backstory="PhD researcher with 10 years experience in tech journalism. Known for finding primary sources.",
#     llm=llm,
#     allow_delegation=False,
#     verbose=True,
#     max_iter=3,
# )
# fact_checker = Agent(
#     role="Fact Checker",
#     goal="Verify every claim against primary sources before it reaches the writer",
#     backstory="Meticulous editor with a background in investigative journalism. Marks each claim CONFIRMED or UNCERTAIN.",
#     llm=llm,
#     allow_delegation=False,
#     verbose=True,
#     max_iter=3,
# )

print("Research agents ready.")

## Section 2 — Research Crew Tasks

Tasks link agents to work items. The `description` is what the agent sees as its assignment. The `expected_output` constrains what a valid result looks like — CrewAI uses this for output validation.

The `{topic}` placeholder is filled at kickoff time via `inputs={"topic": "..."}`. This keeps tasks reusable across different subjects.

The `context=[research_task]` on the fact-check task tells CrewAI to pass the researcher's output as input to the fact checker — even though they run sequentially, `context` makes the dependency explicit.

**TODO 3:** Create `research_task`.

**TODO 4:** Create `fact_check_task` with `context=[research_task]`.

In [ ]:
# TODO 3: Create the research task
# research_task = Task(
#     description="Research {topic} thoroughly. Find 5 key facts with sources.",
#     expected_output="A numbered list of 5 verified facts about {topic}, each with a source citation.",
#     agent=researcher,
# )
research_task = None  # replace with Task(...)

# TODO 4: Create the fact-check task
# The context parameter passes research_task's output into this task automatically.
# fact_check_task = Task(
#     description="Verify all claims in the research. Mark each fact CONFIRMED or UNCERTAIN.",
#     expected_output="A verified version of the research with CONFIRMED or UNCERTAIN next to each fact.",
#     agent=fact_checker,
#     context=[research_task],
# )
fact_check_task = None  # replace with Task(...)

# === SOLUTION (reveal only if stuck) ===
# research_task = Task(
#     description="Research {topic} thoroughly. Find 5 key facts with sources.",
#     expected_output="A numbered list of 5 verified facts about {topic}, each with a source citation.",
#     agent=researcher,
# )
# fact_check_task = Task(
#     description="Verify all claims in the research. Mark each fact CONFIRMED or UNCERTAIN.",
#     expected_output="A verified version of the research with CONFIRMED or UNCERTAIN next to each fact.",
#     agent=fact_checker,
#     context=[research_task],
# )

print("Research tasks ready.")

In [ ]:
# Assemble the research crew (sequential — given)
research_crew = Crew(
    agents=[researcher, fact_checker],
    tasks=[research_task, fact_check_task],
    process=Process.sequential,
    verbose=True,
)
print("Research crew assembled.")

## Section 3 — Writing Crew Agents

The writing crew uses `Process.hierarchical`, which changes how tasks are assigned:

- In **sequential** mode, each task is assigned to its `agent` in order.
- In **hierarchical** mode, the `manager_agent` reads all tasks and delegates them dynamically — it can reassign, request revisions, or combine outputs before declaring the crew's work done.

This means `allow_delegation=True` is important for the writer and editor: the manager needs to be able to redirect work to them.

**TODO 5:** Create the `writer` agent.

**TODO 6:** Create the `editor` agent.

**TODO 7:** Create the `manager_agent`. This agent **must** have `allow_delegation=True` — it is the crew's decision-maker.

In [ ]:
# TODO 5: Create the writer agent
# writer = Agent(
#     role="Content Writer",
#     goal="Write engaging, accurate content based on verified research",
#     backstory="Award-winning science communicator who turns dense research into clear, compelling prose.",
#     llm=llm,
#     allow_delegation=True,
#     verbose=True,
# )
writer = None  # replace with Agent(...)

# TODO 6: Create the editor agent
# editor = Agent(
#     role="Senior Editor",
#     goal="Ensure every piece is clear, accurate, and publication-ready",
#     backstory="25 years in tech publishing. Cuts filler, fixes ambiguity, and demands precise language.",
#     llm=llm,
#     allow_delegation=True,
#     verbose=True,
# )
editor = None  # replace with Agent(...)

# TODO 7: Create the manager agent
# manager_agent = Agent(
#     role="Content Director",
#     goal="Deliver publication-ready content by coordinating writer and editor",
#     backstory="Experienced content director who ensures every piece meets quality standards before publishing.",
#     llm=llm,
#     allow_delegation=True,
#     verbose=True,
# )
manager_agent = None  # replace with Agent(...)

# === SOLUTION (reveal only if stuck) ===
# writer = Agent(
#     role="Content Writer",
#     goal="Write engaging, accurate content based on verified research",
#     backstory="Award-winning science communicator who turns dense research into clear, compelling prose.",
#     llm=llm,
#     allow_delegation=True,
#     verbose=True,
# )
# editor = Agent(
#     role="Senior Editor",
#     goal="Ensure every piece is clear, accurate, and publication-ready",
#     backstory="25 years in tech publishing. Cuts filler, fixes ambiguity, and demands precise language.",
#     llm=llm,
#     allow_delegation=True,
#     verbose=True,
# )
# manager_agent = Agent(
#     role="Content Director",
#     goal="Deliver publication-ready content by coordinating writer and editor",
#     backstory="Experienced content director who ensures every piece meets quality standards before publishing.",
#     llm=llm,
#     allow_delegation=True,
#     verbose=True,
# )

print("Writing agents ready.")

## Section 4 — Writing Crew Tasks and Hierarchical Crew

Writing tasks use `output_file=` so the crew persists results to disk. This is useful in pipelines where the next stage reads from a file rather than an in-memory object.

**TODO 8:** Create `write_task` and `edit_task`. Both should use `output_file=` to save results.

In [ ]:
# TODO 8: Create the writing tasks
# write_task = Task(
#     description=(
#         "Using the verified research provided, write a 200-word article about {topic}. "
#         "Be accurate, engaging, and cite the facts from the research."
#     ),
#     expected_output="A 200-word article with an introduction, body, and conclusion.",
#     agent=writer,
#     output_file="draft_article.txt",
# )
write_task = None  # replace with Task(...)

# edit_task = Task(
#     description="Edit the draft article for clarity, accuracy, and flow. Fix any awkward sentences.",
#     expected_output="A polished, publication-ready article of at least 150 words.",
#     agent=editor,
#     context=[write_task],
#     output_file="final_article.txt",
# )
edit_task = None  # replace with Task(...)

# === SOLUTION (reveal only if stuck) ===
# write_task = Task(
#     description=(
#         "Using the verified research provided, write a 200-word article about {topic}. "
#         "Be accurate, engaging, and cite the facts from the research."
#     ),
#     expected_output="A 200-word article with an introduction, body, and conclusion.",
#     agent=writer,
#     output_file="draft_article.txt",
# )
# edit_task = Task(
#     description="Edit the draft article for clarity, accuracy, and flow. Fix any awkward sentences.",
#     expected_output="A polished, publication-ready article of at least 150 words.",
#     agent=editor,
#     context=[write_task],
#     output_file="final_article.txt",
# )

print("Writing tasks ready.")

In [ ]:
# Assemble the writing crew (hierarchical — given)
writing_crew = Crew(
    agents=[writer, editor],
    tasks=[write_task, edit_task],
    process=Process.hierarchical,
    manager_agent=manager_agent,
    verbose=True,
)
print("Writing crew assembled.")

## Section 5 — Flow with Quality Gate

CrewAI Flows connect crews and logic into an event-driven pipeline using decorators:

| Decorator | Fires when... |
|---|---|
| `@start()` | The flow is kicked off |
| `@listen(step)` | The named step completes |
| `@router(step)` | The named step completes — return value is used as the next route name |

`@router` lets you branch: the string you return from the router method is matched against `@listen("route_name")` decorators downstream.

**TODO 9:** Implement all five methods of `ContentPipeline`. The class skeleton and docstrings tell you exactly what each method should do.

Note: `self.state` is a dict-like object that persists across flow steps — use it to pass inputs between steps.

In [ ]:
class ContentPipeline(Flow):
    """End-to-end content production pipeline."""

    @start()
    def research(self):
        """Kick off the research crew with the topic from self.state."""
        # TODO: call research_crew.kickoff(inputs={"topic": self.state.get("topic", "AI agents")})
        # and return the result
        pass

    @listen(research)
    def write(self, research_output):
        """Kick off the writing crew, passing research_output as context."""
        # TODO: call writing_crew.kickoff(inputs={
        #     "topic": self.state.get("topic", "AI agents"),
        #     "research": str(research_output),
        # })
        # and return the result
        pass

    @router(write)
    def quality_gate(self, draft):
        """Route to 'publish' if the draft is long enough, else 'revise'."""
        # TODO: return "publish" if len(str(draft).split()) > 150 else "revise"
        pass

    @listen("publish")
    def publish(self, content):
        """Print the final article."""
        print("\n" + "=" * 60)
        print("PUBLISHED")
        print("=" * 60)
        print(content)
        return content

    @listen("revise")
    def revise(self, short_draft):
        """Request a revision when the draft is too short."""
        print("\nDraft too short — requesting revision from writing crew...")
        # Re-run writing crew with explicit length instruction
        revised = writing_crew.kickoff(inputs={
            "topic": self.state.get("topic", "AI agents"),
            "research": "Please expand the previous draft to at least 200 words.",
        })
        print("\n" + "=" * 60)
        print("PUBLISHED (after revision)")
        print("=" * 60)
        print(revised)
        return revised

# === SOLUTION (reveal only if stuck) ===
# class ContentPipeline(Flow):
#     @start()
#     def research(self):
#         return research_crew.kickoff(inputs={"topic": self.state.get("topic", "AI agents")})
#
#     @listen(research)
#     def write(self, research_output):
#         return writing_crew.kickoff(inputs={
#             "topic": self.state.get("topic", "AI agents"),
#             "research": str(research_output),
#         })
#
#     @router(write)
#     def quality_gate(self, draft):
#         return "publish" if len(str(draft).split()) > 150 else "revise"
#
#     @listen("publish")
#     def publish(self, content):
#         print("\n" + "=" * 60)
#         print("PUBLISHED")
#         print("=" * 60)
#         print(content)
#         return content
#
#     @listen("revise")
#     def revise(self, short_draft):
#         print("\nDraft too short — requesting revision from writing crew...")
#         revised = writing_crew.kickoff(inputs={
#             "topic": self.state.get("topic", "AI agents"),
#             "research": "Please expand the previous draft to at least 200 words.",
#         })
#         print(revised)
#         return revised

print("ContentPipeline class defined.")

## Section 6 — Run the Pipeline and Compare Token Costs

After the pipeline finishes, `usage_metrics` on each crew object shows how many tokens were consumed at each stage. This is critical in production:

- **Research crew** (sequential, no delegation) should be lean — each agent does exactly one task.
- **Writing crew** (hierarchical) will cost more because the manager agent reads all task outputs and may make additional LLM calls for delegation decisions.

Run the cell and compare. If the writing crew is more than 3x the research crew, the manager's system prompt may be too verbose, or the tasks may need tighter `expected_output` constraints.

In [ ]:
pipeline = ContentPipeline()
result = pipeline.kickoff(inputs={"topic": "The future of multi-agent AI systems"})

print("\n" + "=" * 60)
print("USAGE METRICS")
print("=" * 60)
print("Research crew:", research_crew.usage_metrics)
print("Writing crew: ", writing_crew.usage_metrics)

In [ ]:
# Parse metrics for comparison
def summarise_metrics(label, metrics):
    if metrics is None:
        print(f"{label}: no metrics available")
        return
    try:
        total_tokens = getattr(metrics, "total_tokens", None)
        prompt_tokens = getattr(metrics, "prompt_tokens", None)
        completion_tokens = getattr(metrics, "completion_tokens", None)
        print(f"{label}:")
        print(f"  Total tokens:      {total_tokens}")
        print(f"  Prompt tokens:     {prompt_tokens}")
        print(f"  Completion tokens: {completion_tokens}")
    except Exception as e:
        print(f"{label} metrics error: {e}")
        print(f"  Raw: {metrics}")

summarise_metrics("Research crew", research_crew.usage_metrics)
print()
summarise_metrics("Writing crew", writing_crew.usage_metrics)

## Reflection Questions

1. **Process choice:** The research crew uses `Process.sequential` and the writing crew uses `Process.hierarchical`. What would break if you swapped them? When would you want a hierarchical research crew?

2. **Quality gate threshold:** The router uses 150 words as the pass/fail threshold. What are the failure modes of this heuristic? What would a better quality signal look like?

3. **Token cost:** Look at your usage metrics. Which crew consumed more tokens? Does the cost make sense given the work each crew did?

4. **Flow state:** `self.state` persists across all flow steps. If you wanted to track how many revision loops have occurred, how would you store and increment that counter?

5. **Extending the pipeline:** How would you add a third crew — a SEO Optimiser — that runs after publish and returns a keyword-tagged version of the article? Where in the flow would you hook it in?

---

**Congratulations — you have completed the framework elective modules.** You can now:
- Build nested chat pipelines in AutoGen / AG2 (Module 19)
- Design multi-crew workflows with sequential and hierarchical processes in CrewAI (Module 20)
- Wire event-driven flow control with `@start`, `@listen`, and `@router`